This file is used to merged the features from balance sheet, cashflow and financial data

In [1]:
import pandas as pd
from pathlib import Path
from functools import reduce
from pandas.errors import EmptyDataError
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA


def empty_statement_df(statement_name):
    return pd.DataFrame(columns=["company", "date", statement_name])


# Function to convert a statement CSV file into a DataFrame with company, date, and metrics as a dictionary.
# Empty Yahoo Finance exports are allowed; they should not make us drop the whole ticker.
def statement_to_dict_df(file, statement_name):
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    if not Path(file).exists():
        return empty_statement_df(statement_name)

    try:
        df = pd.read_csv(file)
    except EmptyDataError:
        return empty_statement_df(statement_name)

    # Some failed downloads are saved as a one-column CSV containing only "".
    if df.empty or len(df.columns) < 2:
        return empty_statement_df(statement_name)

    metric_col = df.columns[0]
    rows = []

    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows, columns=["company", "date", statement_name])


Get tickers from the balance sheet folder

In [2]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"

# Get the unique tickers from the balance sheet directory
tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)

print(f"Found {len(tickers)} companies")

Found 1025 companies


In [ ]:
master_rows = []
skipped_tickers = []

# Iterate over each ticker and merge every statement that has usable date columns.
# This keeps partial data instead of dropping the whole ticker when one statement is empty.
for ticker in sorted(tickers):
    statement_files = {
        "balancesheet": BALANCE_SHEET_DIR / f"{ticker}_balancesheet.csv",
        "cashflow": CASH_FLOW_DIR / f"{ticker}_cashflow.csv",
        "financials": FINANCIALS_DIR / f"{ticker}_financials.csv",
    }

    statement_dfs = {
        name: statement_to_dict_df(file, name)
        for name, file in statement_files.items()
    }

    nonempty_dfs = [df for df in statement_dfs.values() if not df.empty]

    if not nonempty_dfs:
        skipped_tickers.append(ticker)
        print(f"Skipped {ticker}: no usable statement dates")
        continue

    company_df = reduce(
        lambda left, right: left.merge(right, on=["company", "date"], how="outer"),
        nonempty_dfs
    )

    for statement_name, statement_df in statement_dfs.items():
        if statement_name not in company_df.columns:
            company_df[statement_name] = pd.NA
        company_df[f"has_{statement_name}"] = not statement_df.empty

    master_rows.append(company_df)
    available = [name for name, df in statement_dfs.items() if not df.empty]
    print(f"Processed {ticker}: {', '.join(available)}")

print()
print(f"Processed tickers: {len(master_rows)}")
print(f"Skipped tickers: {len(skipped_tickers)}")


Processed 001390.SZ: balancesheet, cashflow, financials
Processed 0020.HK: balancesheet
Processed 002058.SZ: balancesheet, cashflow, financials
Processed 002273.SZ: balancesheet, cashflow, financials
Processed 002485.SZ: balancesheet, cashflow, financials
Processed 002623.SZ: balancesheet, cashflow, financials
Processed 002762.SZ: balancesheet, cashflow, financials
Processed 005930.KS: balancesheet, cashflow, financials
Processed 014950.KQ: balancesheet, cashflow, financials
Processed 0547.HK: balancesheet
Processed 0700.HK: balancesheet, financials
Processed 0888.HK: balancesheet
Processed 0905.HK: balancesheet
Processed 0992.HK: balancesheet, financials
Skipped 0DQP.L: no usable statement dates
Processed 0GJ.F: balancesheet, cashflow, financials
Processed 0M5.HA: balancesheet, cashflow, financials
Processed 0M5.MU: balancesheet, cashflow, financials
Skipped 0P00002A83.F: no usable statement dates
Skipped 0P00008SWX.SI: no usable statement dates
Skipped 0P0000HN4R.F: no usable stateme

In [ ]:
import pandas as pd

if not master_rows:
    raise ValueError("No usable statement data was found. Check the statement CSV folders.")

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"], errors="coerce")
dataset = dataset.dropna(subset=["date"])

# Create a quarter column based on the statement date.
dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

for statement_name in ["balancesheet", "cashflow", "financials"]:
    if statement_name not in dataset.columns:
        dataset[statement_name] = [{} for _ in range(len(dataset))]
    else:
        dataset[statement_name] = dataset[statement_name].apply(
            lambda x: x if isinstance(x, dict) else {}
        )

bs_features = pd.json_normalize(dataset["balancesheet"]).add_prefix("bs_")
cf_features = pd.json_normalize(dataset["cashflow"]).add_prefix("cf_")
fin_features = pd.json_normalize(dataset["financials"]).add_prefix("fin_")

availability_cols = [
    col for col in ["has_balancesheet", "has_cashflow", "has_financials"]
    if col in dataset.columns
]

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"] + availability_cols].reset_index(drop=True),
        bs_features.reset_index(drop=True),
        cf_features.reset_index(drop=True),
        fin_features.reset_index(drop=True),
    ],
    axis=1,
)

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

print(f"Rows: {len(features_df)}")
print(f"Companies: {features_df['company'].nunique()}")
print(features_df[availability_cols].sum() if availability_cols else "No availability flags")
print(features_df.columns)


Index(['company', 'date', 'quarter', 'bs_Cash Equivalents',
       'bs_Cash Financial', 'bs_Ordinary Shares Number', 'bs_Share Issued',
       'bs_Tangible Book Value', 'bs_Invested Capital', 'bs_Working Capital',
       ...
       'fin_Excise Taxes', 'fin_Rent Expense Supplemental',
       'fin_Rent And Landing Fees', 'fin_Other Taxes',
       'fin_Insurance And Claims', 'fin_Loss Adjustment Expense',
       'fin_Net Policyholder Benefits And Claims',
       'fin_Policyholder Benefits Gross', 'fin_Policyholder Benefits Ceded',
       'fin_Net Income From Tax Loss Carryforward'],
      dtype='str', length=342)


Compare the date from the dataset to and labelling the layoff

In [ ]:
import ast
import pandas as pd


def parse_layoff_dates(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        parsed = [value]
    if not isinstance(parsed, list):
        parsed = [parsed]
    return parsed


dataset = pd.read_csv(MERGED_DATA["MERGED_OUTPUT_CSV_PATH"])

# Use the combined cleaned file so labels come from the same ticker universe used for the larger dataset.
# Its dates column stores lists as strings, so we parse and explode it first.
layoff_df = pd.read_csv(DATA_DIR / "cleaned" / "cleaned_layoffs_with_tickers_combined.csv")
layoff_df = layoff_df.dropna(subset=["Ticker", "dates"])
layoff_df["dates"] = layoff_df["dates"].apply(parse_layoff_dates)
layoff_df = layoff_df.explode("dates")
layoff_df["dates"] = pd.to_datetime(layoff_df["dates"], errors="coerce")
layoff_df = layoff_df.dropna(subset=["dates"])
layoff_df["quarter"] = layoff_df["dates"].dt.to_period("Q").astype(str)

dataset["quarter"] = pd.PeriodIndex(dataset["quarter"], freq="Q")
dataset["same_quarter"] = dataset["quarter"].astype(str)
dataset["next_quarter"] = (dataset["quarter"] + 1).astype(str)

layoff_events = set(zip(layoff_df["Ticker"].astype(str), layoff_df["quarter"].astype(str)))

same_quarter_pairs = list(zip(dataset["company"].astype(str), dataset["same_quarter"]))
next_quarter_pairs = list(zip(dataset["company"].astype(str), dataset["next_quarter"]))

# Keep both labels so you can choose the target later.
dataset["label_same_quarter"] = pd.Series(same_quarter_pairs).isin(layoff_events).astype(int).values
dataset["label_next_quarter"] = pd.Series(next_quarter_pairs).isin(layoff_events).astype(int).values
dataset["label_same_or_next_quarter"] = (
    (dataset["label_same_quarter"] == 1) |
    (dataset["label_next_quarter"] == 1)
).astype(int)

# Backwards-compatible default label. Change this if you only want next-quarter prediction.
dataset["label"] = dataset["label_same_or_next_quarter"]

dataset["quarter"] = dataset["quarter"].astype(str)

print("Label counts:")
print(dataset[["label_same_quarter", "label_next_quarter", "label_same_or_next_quarter", "label"]].sum())
print(dataset["label"].value_counts())

positive = dataset[dataset["label"] == 1]
print(
    positive[
        ["company", "quarter", "same_quarter", "next_quarter", "label_same_quarter", "label_next_quarter"]
    ].head(20)
)

dataset.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)


str
0    2024Q3
1    2024Q4
2    2025Q1
3    2025Q2
4    2025Q3
Name: quarter, dtype: str
  quarter target_quarter
0  2024Q3         2024Q4
1  2024Q4         2025Q1
2  2025Q1         2025Q2
3  2025Q2         2025Q3
4  2025Q3         2025Q4
label
0    3589
1      94
Name: count, dtype: int64
    company quarter target_quarter
3      MBLY  2025Q2         2025Q3
15     FIVN  2024Q4         2025Q1
49     INTC  2025Q1         2025Q2
51     INTC  2025Q3         2025Q4
84        U  2025Q3         2025Q4
119    ISRG  2024Q3         2024Q4
129    PAYO  2025Q2         2025Q3
139   0GJ.F  2026Q1         2026Q2
430    EXPE  2025Q4         2026Q1
486     DOX  2025Q3         2025Q4
591    CRWD  2025Q2         2025Q3
651    PTON  2025Q2         2025Q3
662    SONO  2025Q1         2025Q2
665    SONO  2025Q4         2026Q1
758    GEMI  2025Q1         2025Q2
777     IBM  2024Q3         2024Q4
945    META  2025Q3         2025Q4
946    META  2025Q4         2026Q1
951    MTCH  2025Q2         2025Q3
956    C